# NovaCart E-commerce Analytics
## Notebook 02 — Data Cleaning

This notebook continues directly from `01_data_quality.ipynb`.

### Objectives

We will clean and standardize:

- Missing values
- Duplicate transaction rows
- Invalid dates
- Inconsistent capitalization
- Leading/trailing whitespace
- City misspellings
- Payment/channel naming inconsistencies
- Negative quantities
- Zero prices
- Missing product IDs
- Return-data inconsistencies
- Marketing missing values

We will also:

- Preserve all files in `data/raw/`
- Save cleaned files into `data/cleaned/`
- Recalculate financial columns where necessary
- Validate primary keys and foreign keys after cleaning
- Produce a final cleaning summary

## 1. Import libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Define project paths

In [2]:
PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Raw:", RAW_DIR.resolve())
print("Cleaned:", CLEAN_DIR.resolve())

Raw: C:\Users\Didar\Desktop\Ecommerce_Analytics_Project\data\raw
Cleaned: C:\Users\Didar\Desktop\Ecommerce_Analytics_Project\data\cleaned


## 3. Load raw datasets

We always load fresh copies from the raw folder so the source data remains unchanged.

In [3]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")
returns = pd.read_csv(RAW_DIR / "returns.csv")
marketing = pd.read_csv(RAW_DIR / "marketing.csv")

print("Raw datasets loaded.")

Raw datasets loaded.


## 4. Record starting row counts

This lets us explain exactly what changed during cleaning.

In [4]:
starting_rows = {
    "customers": len(customers),
    "products": len(products),
    "orders": len(orders),
    "order_items": len(order_items),
    "returns": len(returns),
    "marketing": len(marketing)
}

pd.DataFrame(
    starting_rows.items(),
    columns=["Dataset", "Starting_Rows"]
)

,Dataset,Starting_Rows
0,customers,25000
1,products,1200
2,orders,112000
3,order_items,227961
4,returns,16185
5,marketing,360


# Part A — Customers Cleaning

## 5. Inspect customer issues

In [5]:
print("Missing values:")
display(customers.isna().sum())

print("\nTop city values:")
display(customers["City"].value_counts(dropna=False).head(30))

print("\nEmail opt-in values:")
display(customers["Email_Opt_In"].value_counts(dropna=False))

Missing values:


Customer_ID           0
Customer_Name         0
City                  0
State                 0
Customer_Segment      0
Signup_Date           0
Email_Opt_In        150
dtype: int64


Top city values:


City
Other           7370
New York        2304
Los Angeles     1890
Chicago         1311
Houston         1255
Phoenix          786
Philadelphia     786
Dallas           766
San Antonio      697
Miami            692
Denver           688
Atlanta          682
Austin           649
Boston           642
San Diego        633
Seattle          619
Charlotte        580
Orlando          565
Portland         518
Columbus         506
Nashville        486
OTHER             76
 Other            73
New Yrok          25
Huston            25
Seatle            25
Los Angles        25
Chcago            25
 New York         22
NEW YORK          22
Name: count, dtype: int64


Email opt-in values:


Email_Opt_In
Yes    17835
No      7015
NaN      150
Name: count, dtype: int64

## 6. Standardize customer text

We will:

- strip leading/trailing spaces
- standardize city casing
- repair known city misspellings
- standardize segment and email opt-in values

In [6]:
text_cols = ["Customer_Name", "City", "State", "Customer_Segment", "Email_Opt_In"]

for col in text_cols:
    customers[col] = customers[col].astype("string").str.strip()

customers["City"] = customers["City"].str.title()
customers["Customer_Segment"] = customers["Customer_Segment"].str.title()
customers["Email_Opt_In"] = customers["Email_Opt_In"].str.title()

city_corrections = {
    "New Yrok": "New York",
    "Los Angles": "Los Angeles",
    "Chcago": "Chicago",
    "Huston": "Houston",
    "Seatle": "Seattle"
}

customers["City"] = customers["City"].replace(city_corrections)

display(customers["City"].value_counts().head(25))

City
Other           7519
New York        2373
Los Angeles     1947
Chicago         1355
Houston         1298
Phoenix          806
Philadelphia     801
Dallas           778
San Antonio      710
Denver           702
Miami            700
Atlanta          693
Austin           660
Seattle          654
Boston           650
San Diego        645
Charlotte        588
Orlando          578
Portland         530
Columbus         515
Nashville        498
Name: count, dtype: Int64

## 7. Handle missing customer values

For `Email_Opt_In`, missing values are treated as `Unknown` instead of assuming Yes/No.

In [7]:
customers["Email_Opt_In"] = customers["Email_Opt_In"].fillna("Unknown")

print(customers["Email_Opt_In"].value_counts(dropna=False))

Email_Opt_In
Yes        17835
No          7015
Unknown      150
Name: count, dtype: Int64


## 8. Parse customer signup dates

In [8]:
customers["Signup_Date"] = pd.to_datetime(
    customers["Signup_Date"],
    errors="coerce"
)

print("Invalid Signup_Date values:", customers["Signup_Date"].isna().sum())

Invalid Signup_Date values: 0


# Part B — Product Cleaning

## 9. Standardize product text

In [9]:
product_text_cols = ["Product_Name", "Category", "Subcategory", "Brand"]

for col in product_text_cols:
    products[col] = products[col].astype("string").str.strip()

products["Category"] = products["Category"].str.title()
products["Subcategory"] = products["Subcategory"].str.title()

print("Categories:")
display(products["Category"].value_counts())

print("\nMissing brands:")
print(products["Brand"].isna().sum())

Categories:


Category
Electronics          237
Fashion              220
Home & Kitchen       191
Sports & Outdoors    178
Office               135
Beauty               121
Pet Supplies         118
Name: count, dtype: Int64


Missing brands:
15


## 10. Handle missing product brands

A missing brand should not cause rows to be dropped. We label it `Unknown`.

In [10]:
products["Brand"] = products["Brand"].fillna("Unknown")

print("Missing brands after cleaning:", products["Brand"].isna().sum())

Missing brands after cleaning: 0


## 11. Validate product numeric fields

In [11]:
product_numeric_cols = [
    "List_Price",
    "Unit_Cost",
    "Base_Margin",
    "Base_Return_Rate"
]

for col in product_numeric_cols:
    products[col] = pd.to_numeric(products[col], errors="coerce")

print(products[product_numeric_cols].describe())

       List_Price  Unit_Cost  Base_Margin  Base_Return_Rate
count    1,200.00   1,200.00     1,200.00          1,200.00
mean        55.80      36.70         0.36              0.07
std         37.11      27.73         0.10              0.03
min          7.45       3.89         0.18              0.04
25%         31.18      18.52         0.29              0.04
50%         46.84      28.83         0.35              0.07
75%         68.50      45.46         0.42              0.09
max        280.00     227.59         0.62              0.13


# Part C — Orders Cleaning

## 12. Standardize order categories

In [12]:
order_text_cols = [
    "Channel",
    "Payment_Method",
    "Shipping_Method",
    "Order_Status"
]

for col in order_text_cols:
    orders[col] = orders[col].astype("string").str.strip()

orders["Channel"] = orders["Channel"].str.title()
orders["Payment_Method"] = orders["Payment_Method"].str.title()
orders["Shipping_Method"] = orders["Shipping_Method"].str.title()
orders["Order_Status"] = orders["Order_Status"].str.title()

print("Channels:")
display(orders["Channel"].value_counts(dropna=False))

print("\nPayment methods:")
display(orders["Payment_Method"].value_counts(dropna=False))

Channels:


Channel
Organic Search    27019
Paid Search       22617
Social Media      16848
Email             14350
Direct            13408
Marketplace        9994
Referral           7764
Name: count, dtype: Int64


Payment methods:


Payment_Method
Credit Card          46784
Paypal               22575
Debit Card           18001
Apple Pay             8981
Buy Now Pay Later     8950
Google Pay            6709
Name: count, dtype: Int64

## 13. Correct capitalization-sensitive payment labels

`.str.title()` changes some brand names undesirably, so we explicitly restore canonical business labels.

In [13]:
payment_corrections = {
    "Paypal": "PayPal",
    "Apple Pay": "Apple Pay",
    "Google Pay": "Google Pay",
    "Buy Now Pay Later": "Buy Now Pay Later",
    "Credit Card": "Credit Card",
    "Debit Card": "Debit Card"
}

orders["Payment_Method"] = orders["Payment_Method"].replace(payment_corrections)

channel_corrections = {
    "Paid Search": "Paid Search",
    "Organic Search": "Organic Search",
    "Social Media": "Social Media",
    "Email": "Email",
    "Direct": "Direct",
    "Referral": "Referral",
    "Marketplace": "Marketplace"
}

orders["Channel"] = orders["Channel"].replace(channel_corrections)

## 14. Parse order and shipping dates

Invalid text values become `NaT`.

In [14]:
orders["Order_Date"] = pd.to_datetime(
    orders["Order_Date"],
    errors="coerce"
)

orders["Ship_Date"] = pd.to_datetime(
    orders["Ship_Date"],
    errors="coerce"
)

print("Invalid/missing Order_Date:", orders["Order_Date"].isna().sum())
print("Invalid/missing Ship_Date:", orders["Ship_Date"].isna().sum())

Invalid/missing Order_Date: 100
Invalid/missing Ship_Date: 4795


## 15. Handle missing shipping methods

For completed orders with missing shipping method, we label the value `Unknown`.

Cancelled orders do not need a shipping method.

In [15]:
orders.loc[
    orders["Shipping_Method"].isna()
    & orders["Order_Status"].eq("Completed"),
    "Shipping_Method"
] = "Unknown"

orders.loc[
    orders["Shipping_Method"].isna()
    & orders["Order_Status"].eq("Cancelled"),
    "Shipping_Method"
] = "Not Applicable"

print(orders["Shipping_Method"].value_counts(dropna=False))

Shipping_Method
Standard          84908
Express           22497
Same Day           4345
Unknown             238
Not Applicable       12
Name: count, dtype: Int64


## 16. Remove orders with invalid Order_Date

An order without a valid transaction date cannot be reliably used in time-series analysis.

We remove only rows where the original raw value was invalid.

In [16]:
invalid_order_date_rows = orders["Order_Date"].isna().sum()

orders = orders.loc[
    orders["Order_Date"].notna()
].copy()

print("Removed invalid Order_Date rows:", invalid_order_date_rows)
print("Remaining orders:", len(orders))

Removed invalid Order_Date rows: 100
Remaining orders: 111900


# Part D — Order Items Cleaning

## 17. Remove exact duplicate transaction lines

In [17]:
duplicates_before = order_items.duplicated().sum()

order_items = order_items.drop_duplicates().copy()

print("Exact duplicate rows removed:", duplicates_before)
print("Rows remaining:", len(order_items))

Exact duplicate rows removed: 450
Rows remaining: 227511


## 18. Convert numeric columns

In [18]:
item_numeric_cols = [
    "Quantity",
    "Unit_Price",
    "Discount_Pct",
    "Gross_Sales",
    "Discount_Amount",
    "Net_Sales",
    "Cost_Amount",
    "Profit"
]

for col in item_numeric_cols:
    order_items[col] = pd.to_numeric(
        order_items[col],
        errors="coerce"
    )

display(order_items[item_numeric_cols].describe())

,Quantity,Unit_Price,Discount_Pct,Gross_Sales,Discount_Amount,Net_Sales,Cost_Amount,Profit
count,"227,511.00","227,511.00","227,211.00","227,511.00","227,511.00","227,511.00","227,511.00","227,511.00"
mean,1.45,55.40,0.06,80.40,4.65,72.39,50.69,21.70
std,0.80,35.69,0.06,73.89,8.04,69.88,52.54,21.07
min,-1.00,0.00,0.00,7.45,0.00,0.00,0.00,-24.12
25%,1.00,32.64,0.00,37.23,0.00,32.72,21.23,9.56
50%,1.00,47.32,0.05,58.21,1.95,52.80,36.04,15.97
75%,2.00,66.38,0.10,97.62,6.17,88.96,61.76,26.38
max,6.00,280.00,0.30,"1,560.72",214.87,"1,560.72","1,050.96",509.76


## 19. Handle missing product IDs

Rows without `Product_ID` cannot be connected to the product dimension.

We remove them because product-level reporting would otherwise be incorrect.

In [19]:
missing_product_ids = order_items["Product_ID"].isna().sum()

order_items = order_items.loc[
    order_items["Product_ID"].notna()
].copy()

print("Rows removed for missing Product_ID:", missing_product_ids)

Rows removed for missing Product_ID: 150


## 20. Fix invalid quantities and prices

We use the trusted product table to repair these fields.

- Negative/zero quantities are set to 1.
- Zero/missing unit price is restored from `products.List_Price`.

In [20]:
invalid_qty = order_items["Quantity"].isna() | (order_items["Quantity"] <= 0)

print("Invalid quantities before:", invalid_qty.sum())

order_items.loc[invalid_qty, "Quantity"] = 1

price_lookup = products.set_index("Product_ID")["List_Price"]

invalid_price = (
    order_items["Unit_Price"].isna()
    | (order_items["Unit_Price"] <= 0)
)

order_items.loc[
    invalid_price,
    "Unit_Price"
] = order_items.loc[
    invalid_price,
    "Product_ID"
].map(price_lookup)

print("Invalid prices repaired:", invalid_price.sum())

Invalid quantities before: 150
Invalid prices repaired: 100


## 21. Handle missing discounts

If discount is missing, use `0` rather than deleting the sale.

In [21]:
missing_discount = order_items["Discount_Pct"].isna().sum()

order_items["Discount_Pct"] = (
    order_items["Discount_Pct"]
    .fillna(0)
    .clip(lower=0, upper=1)
)

print("Missing discounts filled:", missing_discount)

Missing discounts filled: 300


## 22. Recalculate financial columns

Because quantity, unit price, and discount may have been repaired, we must recalculate the downstream financial fields.

Business formulas:

```text
Gross Sales      = Quantity × Unit Price
Discount Amount  = Gross Sales × Discount %
Net Sales        = Gross Sales - Discount Amount
Cost Amount      = Quantity × Product Unit Cost
Profit           = Net Sales - Cost Amount
```

Cancelled orders will have zero recognized revenue, cost, and profit.

In [22]:
cost_lookup = products.set_index("Product_ID")["Unit_Cost"]

order_items["Gross_Sales"] = (
    order_items["Quantity"] * order_items["Unit_Price"]
)

order_items["Discount_Amount"] = (
    order_items["Gross_Sales"] * order_items["Discount_Pct"]
)

order_items["Net_Sales"] = (
    order_items["Gross_Sales"] - order_items["Discount_Amount"]
)

order_items["Cost_Amount"] = (
    order_items["Quantity"]
    * order_items["Product_ID"].map(cost_lookup)
)

valid_order_status = orders.set_index("Order_ID")["Order_Status"]

order_items["Order_Status"] = (
    order_items["Order_ID"].map(valid_order_status)
)

cancel_mask = order_items["Order_Status"].eq("Cancelled")

order_items.loc[
    cancel_mask,
    ["Discount_Amount", "Net_Sales", "Cost_Amount"]
] = 0

order_items["Profit"] = (
    order_items["Net_Sales"] - order_items["Cost_Amount"]
)

money_cols = [
    "Unit_Price",
    "Gross_Sales",
    "Discount_Amount",
    "Net_Sales",
    "Cost_Amount",
    "Profit"
]

order_items[money_cols] = order_items[money_cols].round(2)

order_items.drop(
    columns=["Order_Status"],
    inplace=True
)

display(order_items.head())

,Order_Item_ID,Order_ID,Product_ID,Quantity,Unit_Price,Discount_Pct,Gross_Sales,Discount_Amount,Net_Sales,Cost_Amount,Profit
0,OI00000001,O0000001,P01120,1,22.55,0.10,22.55,2.26,20.30,13.95,6.35
1,OI00000002,O0000001,P00781,1,26.91,0.00,26.91,0.00,26.91,15.87,11.04
2,OI00000003,O0000002,P00324,1,55.49,0.10,55.49,5.55,49.94,30.60,19.34
3,OI00000004,O0000002,P00283,4,156.92,0.10,627.68,62.77,564.91,413.32,151.59
4,OI00000005,O0000002,P00318,1,54.68,0.05,54.68,2.73,51.95,38.04,13.91


## 23. Remove orphan order-item rows

Because invalid orders were removed earlier, any order items linked to those removed orders must also be removed.

In [23]:
valid_order_ids = set(orders["Order_ID"])

orphan_items = (~order_items["Order_ID"].isin(valid_order_ids)).sum()

order_items = order_items[
    order_items["Order_ID"].isin(valid_order_ids)
].copy()

print("Orphan order-item rows removed:", orphan_items)

Orphan order-item rows removed: 194


# Part E — Returns Cleaning

## 24. Standardize return fields

In [24]:
returns["Return_Reason"] = (
    returns["Return_Reason"]
    .astype("string")
    .str.strip()
    .str.title()
)

returns["Return_Status"] = (
    returns["Return_Status"]
    .astype("string")
    .str.strip()
    .str.title()
)

returns["Return_Status"] = returns["Return_Status"].fillna("Unknown")

display(returns["Return_Reason"].value_counts(dropna=False))
display(returns["Return_Status"].value_counts(dropna=False))

Return_Reason
Not As Expected        4063
Damaged                3962
Changed Mind           2822
Wrong Size             2078
Defective              1215
Wrong Item             1075
Compatibility Issue     970
Name: count, dtype: Int64

Return_Status
Refunded    14898
Pending      1167
Unknown       120
Name: count, dtype: Int64

## 25. Parse return dates

In [25]:
returns["Return_Date"] = pd.to_datetime(
    returns["Return_Date"],
    errors="coerce"
)

invalid_return_dates = returns["Return_Date"].isna().sum()

print("Invalid return dates:", invalid_return_dates)

Invalid return dates: 100


## 26. Repair missing refund amounts

We use the cleaned order-item `Net_Sales` value as a reasonable fallback only where a refund amount is missing.

This preserves the row while keeping the logic auditable.

In [26]:
net_sales_lookup = order_items.set_index(
    "Order_Item_ID"
)["Net_Sales"]

missing_refunds = returns["Refund_Amount"].isna()

returns.loc[
    missing_refunds,
    "Refund_Amount"
] = returns.loc[
    missing_refunds,
    "Order_Item_ID"
].map(net_sales_lookup)

print("Missing refunds repaired:", missing_refunds.sum())

Missing refunds repaired: 120


## 27. Remove invalid/orphan return rows

A return should reference an existing cleaned order item.

Rows with invalid return dates are also removed.

In [27]:
valid_item_ids = set(order_items["Order_Item_ID"])

orphan_return_mask = (
    ~returns["Order_Item_ID"].isin(valid_item_ids)
)

invalid_return_mask = returns["Return_Date"].isna()

print("Orphan return rows:", orphan_return_mask.sum())
print("Invalid-date return rows:", invalid_return_mask.sum())

returns = returns.loc[
    ~orphan_return_mask
    & ~invalid_return_mask
].copy()

Orphan return rows: 32
Invalid-date return rows: 100


# Part F — Marketing Cleaning

## 28. Standardize marketing categories

In [28]:
marketing["Channel"] = (
    marketing["Channel"]
    .astype("string")
    .str.strip()
    .str.title()
)

marketing["Campaign_Type"] = (
    marketing["Campaign_Type"]
    .astype("string")
    .str.strip()
    .str.title()
)

marketing["Month"] = pd.to_datetime(
    marketing["Month"],
    errors="coerce"
)

display(marketing["Channel"].value_counts())
display(marketing["Campaign_Type"].value_counts())

Channel
Paid Search     72
Social Media    72
Email           72
Referral        72
Marketplace     72
Name: count, dtype: Int64

Campaign_Type
Always On      120
Promotion      120
Retargeting    120
Name: count, dtype: Int64

## 29. Repair missing marketing values

For missing `Spend` and `Clicks`, we use the median within the same marketing channel.

This is preferable to using one global median because channels have different economics.

In [29]:
marketing["Spend"] = pd.to_numeric(
    marketing["Spend"],
    errors="coerce"
)

marketing["Clicks"] = pd.to_numeric(
    marketing["Clicks"],
    errors="coerce"
)

marketing["Spend"] = (
    marketing["Spend"]
    .fillna(
        marketing.groupby("Channel")["Spend"]
        .transform("median")
    )
)

marketing["Clicks"] = (
    marketing["Clicks"]
    .fillna(
        marketing.groupby("Channel")["Clicks"]
        .transform("median")
    )
)

print("Missing spend:", marketing["Spend"].isna().sum())
print("Missing clicks:", marketing["Clicks"].isna().sum())

Missing spend: 0
Missing clicks: 0


## 30. Recalculate marketing KPIs

Because clicks may have been repaired, CTR is recalculated.

```text
CTR = Clicks / Impressions
Conversion Rate = Attributed Orders / Clicks
```

In [30]:
marketing["CTR"] = (
    marketing["Clicks"]
    / marketing["Impressions"]
)

marketing["Conversion_Rate"] = (
    marketing["Attributed_Orders"]
    / marketing["Clicks"]
)

marketing["CTR"] = marketing["CTR"].round(4)
marketing["Conversion_Rate"] = marketing["Conversion_Rate"].round(4)
marketing["Spend"] = marketing["Spend"].round(2)

display(marketing.head())

,Campaign_ID,Month,Channel,Campaign_Type,Spend,Impressions,Clicks,Attributed_Orders,CTR,Conversion_Rate
0,M00001,2024-01-01,Paid Search,Always On,"23,762.47",742577,"28,098.00",1186,0.04,0.04
1,M00002,2024-01-01,Paid Search,Promotion,"13,429.63",419675,"16,605.00",727,0.04,0.04
2,M00003,2024-01-01,Paid Search,Retargeting,"9,774.27",305445,"12,393.00",517,0.04,0.04
3,M00004,2024-01-01,Social Media,Always On,"16,191.51",1471955,"28,705.00",616,0.02,0.02
4,M00005,2024-01-01,Social Media,Promotion,"11,320.95",1029177,"16,799.00",441,0.02,0.03


# Part G — Final Validation

## 31. Validate primary keys

In [31]:
pk_checks = {
    "customers": (customers, "Customer_ID"),
    "products": (products, "Product_ID"),
    "orders": (orders, "Order_ID"),
    "order_items": (order_items, "Order_Item_ID"),
    "returns": (returns, "Return_ID"),
    "marketing": (marketing, "Campaign_ID")
}

pk_results = []

for name, (df, key) in pk_checks.items():
    pk_results.append({
        "Dataset": name,
        "Rows": len(df),
        "Unique_Keys": df[key].nunique(dropna=True),
        "Missing_Keys": int(df[key].isna().sum()),
        "Duplicate_Keys": int(df[key].duplicated().sum())
    })

pk_results_df = pd.DataFrame(pk_results)
pk_results_df

,Dataset,Rows,Unique_Keys,Missing_Keys,Duplicate_Keys
0,customers,25000,25000,0,0
1,products,1200,1200,0,0
2,orders,111900,111900,0,0
3,order_items,227167,227167,0,0
4,returns,16054,16054,0,0
5,marketing,360,360,0,0


## 32. Validate foreign keys

In [32]:
fk_results = []

def check_fk(child_name, child_df, child_col, parent_name, parent_df, parent_col):
    bad = (
        child_df[child_col]
        .dropna()
        .isin(parent_df[parent_col])
        .eq(False)
        .sum()
    )

    fk_results.append({
        "Child_Table": child_name,
        "Child_Key": child_col,
        "Parent_Table": parent_name,
        "Missing_References": int(bad)
    })

check_fk("orders", orders, "Customer_ID", "customers", customers, "Customer_ID")
check_fk("order_items", order_items, "Order_ID", "orders", orders, "Order_ID")
check_fk("order_items", order_items, "Product_ID", "products", products, "Product_ID")
check_fk("returns", returns, "Order_Item_ID", "order_items", order_items, "Order_Item_ID")
check_fk("returns", returns, "Order_ID", "orders", orders, "Order_ID")
check_fk("returns", returns, "Product_ID", "products", products, "Product_ID")

fk_results_df = pd.DataFrame(fk_results)
fk_results_df

,Child_Table,Child_Key,Parent_Table,Missing_References
0,orders,Customer_ID,customers,0
1,order_items,Order_ID,orders,0
2,order_items,Product_ID,products,0
3,returns,Order_Item_ID,order_items,0
4,returns,Order_ID,orders,0
5,returns,Product_ID,products,0


## 33. Validate key business rules

In [33]:
validation = []

def add_validation(check, issues):
    validation.append({
        "Check": check,
        "Issues_Remaining": int(issues)
    })

add_validation(
    "Order item Quantity <= 0",
    (order_items["Quantity"] <= 0).sum()
)

add_validation(
    "Order item Unit_Price <= 0",
    (order_items["Unit_Price"] <= 0).sum()
)

add_validation(
    "Discount outside 0-1",
    (
        (order_items["Discount_Pct"] < 0)
        | (order_items["Discount_Pct"] > 1)
    ).sum()
)

expected_profit = (
    order_items["Net_Sales"]
    - order_items["Cost_Amount"]
)

add_validation(
    "Profit formula mismatch",
    (
        ~np.isclose(
            order_items["Profit"],
            expected_profit,
            atol=0.02,
            equal_nan=True
        )
    ).sum()
)

expected_gross = (
    order_items["Quantity"]
    * order_items["Unit_Price"]
)

add_validation(
    "Gross Sales formula mismatch",
    (
        ~np.isclose(
            order_items["Gross_Sales"],
            expected_gross,
            atol=0.02,
            equal_nan=True
        )
    ).sum()
)

add_validation(
    "Exact duplicate order-item rows",
    order_items.duplicated().sum()
)

validation_df = pd.DataFrame(validation)
validation_df

,Check,Issues_Remaining
0,Order item Quantity <= 0,0
1,Order item Unit_Price <= 0,0
2,Discount outside 0-1,0
3,Profit formula mismatch,0
4,Gross Sales formula mismatch,0
5,Exact duplicate order-item rows,0


## 34. Compare raw vs cleaned row counts

In [34]:
ending_rows = {
    "customers": len(customers),
    "products": len(products),
    "orders": len(orders),
    "order_items": len(order_items),
    "returns": len(returns),
    "marketing": len(marketing)
}

row_comparison = pd.DataFrame({
    "Dataset": starting_rows.keys(),
    "Raw_Rows": starting_rows.values(),
    "Cleaned_Rows": ending_rows.values()
})

row_comparison["Rows_Removed"] = (
    row_comparison["Raw_Rows"]
    - row_comparison["Cleaned_Rows"]
)

row_comparison

,Dataset,Raw_Rows,Cleaned_Rows,Rows_Removed
0,customers,25000,25000,0
1,products,1200,1200,0
2,orders,112000,111900,100
3,order_items,227961,227167,794
4,returns,16185,16054,131
5,marketing,360,360,0


# Part H — Save Cleaned Files

## 35. Save all cleaned datasets

These files will be used later by SQL and Power BI.

In [35]:
customers.to_csv(
    CLEAN_DIR / "customers_clean.csv",
    index=False
)

products.to_csv(
    CLEAN_DIR / "products_clean.csv",
    index=False
)

orders.to_csv(
    CLEAN_DIR / "orders_clean.csv",
    index=False
)

order_items.to_csv(
    CLEAN_DIR / "order_items_clean.csv",
    index=False
)

returns.to_csv(
    CLEAN_DIR / "returns_clean.csv",
    index=False
)

marketing.to_csv(
    CLEAN_DIR / "marketing_clean.csv",
    index=False
)

print("Cleaned files saved successfully:")
for file in sorted(CLEAN_DIR.glob("*.csv")):
    print("-", file.name)

Cleaned files saved successfully:
- customers_clean.csv
- marketing_clean.csv
- order_items_clean.csv
- orders_clean.csv
- products_clean.csv
- returns_clean.csv


## 36. Final cleaning summary

In [36]:
cleaning_summary = pd.DataFrame([
    ["Customer text standardized", "Completed"],
    ["City misspellings corrected", "Completed"],
    ["Missing customer opt-in handled", "Completed"],
    ["Product categories standardized", "Completed"],
    ["Missing brands handled", "Completed"],
    ["Order categories standardized", "Completed"],
    ["Invalid order dates removed", "Completed"],
    ["Missing shipping method handled", "Completed"],
    ["Duplicate order items removed", "Completed"],
    ["Invalid quantities repaired", "Completed"],
    ["Zero/missing prices repaired", "Completed"],
    ["Missing Product_ID rows removed", "Completed"],
    ["Financial columns recalculated", "Completed"],
    ["Return data standardized", "Completed"],
    ["Invalid/orphan returns removed", "Completed"],
    ["Marketing missing values repaired", "Completed"],
    ["Marketing KPIs recalculated", "Completed"],
    ["Primary keys validated", "Completed"],
    ["Foreign keys validated", "Completed"]
], columns=["Cleaning_Task", "Status"])

cleaning_summary

,Cleaning_Task,Status
0,Customer text standardized,Completed
1,City misspellings corrected,Completed
2,Missing customer opt-in handled,Completed
3,Product categories standardized,Completed
4,Missing brands handled,Completed
5,Order categories standardized,Completed
6,Invalid order dates removed,Completed
7,Missing shipping method handled,Completed
8,Duplicate order items removed,Completed
9,Invalid quantities repaired,Completed


# Conclusion

The cleaned data is now ready for the next stages of the project.

Cleaned files are located in:

```text
data/cleaned/
```

The next recommended notebook is:

```text
03_exploratory_analysis.ipynb
```

In that notebook we will analyze:

- Revenue and profit trends
- Top and bottom categories
- Product profitability
- Customer purchasing behavior
- New vs repeat customers
- Sales by channel
- Discount impact
- Returns and refund trends
- Regional performance
- Marketing efficiency

After EDA, we will load the cleaned files into **SQLite** and solve business questions with SQL before building the Power BI dashboard.